In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from scipy.stats import lognorm, kstest

import sys
from pathlib import Path
sys.path.append(str(Path("../src").resolve()))

In [ ]:
df = pd.read_parquet("../data/processed/whatsapp-20250329-141239-processed.parquet")

df

In [ ]:
# Plot histogram
plt.figure(figsize=(14, 5))
sns.histplot(
    data=df,
    x="message_length",
    bins=100, 
    stat="probability",  # y-as = kans, niet frequentie
    color="skyblue",
    edgecolor="black"
)

plt.xlabel("Aantal tekens per bericht")
plt.ylabel("Kans")
plt.title("Kansverdeling van berichtlengte in de familiechat (histogram)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Alleen positieve lengtes
message_lengths = df["message"].str.len()
positive_lengths = message_lengths[message_lengths > 0]

# Maak bins: 0–200 in stappen van 1 + alles daarboven in 1 bin
bins = list(range(0, 200, 3)) + [positive_lengths.max()]

# Lognormale fit
shape, loc, scale = lognorm.fit(positive_lengths, floc=0)
x = np.linspace(positive_lengths.min(), positive_lengths.max(), 1000)
pdf = lognorm.pdf(x, shape, loc=loc, scale=scale)

# Schaal naar percentage: pdf * 100 * binbreedte
bin_width = np.diff(bins)[0]  # bijvoorbeeld 1
pdf_scaled = pdf * 100 * bin_width

# Plot histogram met aangepaste bins
plt.figure(figsize=(14, 5))
sns.histplot(
    positive_lengths,
    bins=bins,
    stat="percent",
    color="skyblue",
    edgecolor="black",
    label="Data"
)

# Plot fit
plt.plot(x, pdf_scaled, "r", lw=2, label="Lognormale fit")
plt.xlim(0, 200)
xticks = list(range(0, 200, 20)) + [200]
xtick_labels = [str(x) for x in xticks[:-1]] + ['200+']
plt.xticks(xticks, xtick_labels)

# Extra labels
plt.xlabel("Aantal tekens per bericht")
plt.ylabel("Kans in %")
plt.title("Lognormale fit op berichtlengte in de familiechat")
plt.legend()
plt.tight_layout()
plt.show()



### 🗨️ Berichtlengte in de familiechat

De meeste berichten in de familiechat zijn kort en krachtig: zo'n **3 tot 4% van de berichten bevat maar 5 tot 10 tekens**. Dat zie je aan de piek links in de grafiek. Denk aan dingen als *“oké”*, of gewoon één emoji.

De kans op langere berichten daalt daarna gestaag, maar verdwijnt niet. Er zijn ook flink wat langere berichten, en dat past bij de familiechat waar soms een verhaal wordt getypt, een krantenartikel wordt doorgestuurd of een vakantie-update wordt gedeeld.

De rode lijn is een **lognormale verdeling**, en die past, lijkt mij, goed:

- het model voorspelt de scherpe piek bij korte berichten,  
- en laat ook zien dat langere berichten nog wel voorkomen, maar met steeds kleinere kans.

Die uitschieter aan het eind? Dat zijn berichten van **200+ tekens**, zeldzaam, maar wel degelijk aanwezig. 


In [ ]:
plt.figure(figsize=(12, 2))
sns.boxplot(x=df["message_length"])
plt.title("Boxplot van berichtlengte")
plt.show()


In [ ]:
# Sorteer op lengte om de langste berichten te zien
longest_msgs = df.sort_values(by="message_length", ascending=False).head(10)
display(longest_msgs[["timestamp", "author", "message_length", "message"]])

In [ ]:
plt.figure(figsize=(8, 2))
sns.boxplot(x=df["message_length"], color="lightblue")
plt.xlim(0, 500)
plt.xlabel("Aantal tekens per bericht (afgekapt op 500)")
plt.title("Boxplot van berichtlengte (truncatie op 500)")
plt.tight_layout()
plt.show()


In [ ]:
# Bekijk de verdeling
top_lengtes = df["message_length"].value_counts().sort_values(ascending=False)
print("Top meest voorkomende berichtlengtes:\n", top_lengtes.head())

# Filter op de uitschieter-lengte, bijvoorbeeld 25
uitschieter_waarde = top_lengtes.idxmax()  # pakt de meest voorkomende lengte automatisch
uitschieter_waarde
df[df["message_length"] == 20][["timestamp", "author", "message"]]

In [ ]:
# Zorg dat je de message_length kolom hebt
df['message_length'] = df['message'].astype(str).str.len()
positive_lengths = df['message_length'][df['message_length'] > 0]

# Fit een lognormale verdeling op de data
shape, loc, scale = lognorm.fit(positive_lengths, floc=0)

# Voer de KS-test uit
ks_statistic, p_value = kstest(positive_lengths, 'lognorm', args=(shape, loc, scale))

# Resultaat printen
print(f"KS-statistic: {ks_statistic:.4f}")
print(f"P-waarde: {p_value:.4f}")

# Interpretatie
if p_value < 0.05:
    print("De verdeling wijkt erg af van een lognormale verdeling.")
else:
    print("De verdeling wijkt niet erg af van een lognormale verdeling.")
